## Testing gpu-acelerated curbd on our data

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
sys.path.append("../../")

import pyaldata as pyal
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

from tools.reports.report_initial import run_initial_report
from tools.params import Params, colors
from tools.dsp.preprocessing import preprocess
import tools.viz.mean_firing as firing
import tools.viz.dimensionality as dim
import tools.viz.utilityTools as vizutils
import tools.decoding.rrr as rrr
import tools.decoding.decodeTools as decutils
import tools.dataTools as dt
from tools.curbd import curbd_gpu_v2
import cupy as cp

# 20th of March

In [ ]:
# Files 
session = 'M062_2025_03_20_14_00'
data_dir = f"/data/bnd-data/raw/M062/{session}"

areas=["MOp", "SSp", "CP", "VAL"]
df = pyal.load_pyaldata(data_dir)

In [ ]:
df_ = preprocess(df, only_trials=False, repair_time_varying_fields=['MotSen1_X', 'MotSen1_Y'])

### Get trials

In [ ]:
activity_ = []
for sol_dir in range(12):
    df_trials = pyal.select_trials(df_, df_.values_Sol_direction == 1)
    df_trials = df_trials.iloc[:-1] 
    trial_length = df_trials.MOp_rates.values[0].shape[0]

    activity = []
    neurons = []
    for area in areas:
        neurons.append(df_trials[f"{area}_rates"][0].shape[-1])
        activity.append(pyal.concat_trials(df_trials, f"{area}_rates").T)

    # Activity
    activity = np.concatenate(activity)
    activity_.append(activity[np.newaxis, :, :])

activity_ = np.concatenate(activity_)

# Regions
regions = []

start_idx = 0
for area, size in zip(areas, neurons):
    regions.append([area, np.arange(start_idx, start_idx + size)])
    start_idx += size  # Update the starting index for the next region

activity_.shape

In [ ]:
# Reshape
activity = activity.reshape(activity.shape[0], activity.shape[1] // trial_length, trial_length)  # Shape (N, M, tr)
activity = activity.transpose(1, 0, 2)  # Shape (M, N, T)
activity = activity[:50, :, :]

activity.shape

### Train gpu curbd

In [ ]:
mempool = cp.get_default_memory_pool()
pinned_mempool = cp.get_default_pinned_memory_pool()


mempool.free_all_blocks()
pinned_mempool.free_all_blocks()
print(mempool.used_bytes())              # 0
print(mempool.total_bytes())             # 0
print(pinned_mempool.n_free_blocks())    # 0

a_cpu = np.ndarray(100, dtype=np.float32)

a = cp.array(a_cpu)
print(a.nbytes)                          # 400
print(mempool.used_bytes())              # 512
print(mempool.total_bytes())             # 512
print(pinned_mempool.n_free_blocks())    # 1

In [ ]:
# When the array goes out of scope, the allocated device memory is released
# and kept in the pool for future reuse.
a = None  # (or `del a`)
print(mempool.used_bytes())              # 0
print(mempool.total_bytes())             # 512
print(pinned_mempool.n_free_blocks())    # 1


In [ ]:
# You can clear the memory pool by calling `free_all_blocks`.
mempool.free_all_blocks()
pinned_mempool.free_all_blocks()
print(mempool.used_bytes())              # 0
print(mempool.total_bytes())             # 0
print(pinned_mempool.n_free_blocks())    # 0

In [ ]:
activity_.shape

In [ ]:
# %%time

gcurbd = curbd_gpu_v2.gCURBD(
    dt_data=df_.bin_size[0],
    dt_factor=3,
    regions=regions,
    train_epochs=1
)

gcurbd.fit(activity_)


In [ ]:
del gcurbd

In [ ]:
print(mempool.used_bytes())              # 0
print(mempool.total_bytes())             # 512
print(pinned_mempool.n_free_blocks())    # 1

In [ ]:
# You can clear the memory pool by calling `free_all_blocks`.
mempool.free_all_blocks()
pinned_mempool.free_all_blocks()
print(mempool.used_bytes())              # 0
print(mempool.total_bytes())             # 0
print(pinned_mempool.n_free_blocks())    # 0

In [ ]:

from tools.curbd import curbd


model = curbd.trainMultiRegionRNN(np.squeeze(activity_[:1, :, :]),
                                  dtData=df_.bin_size[0],
                                  dtFactor=5,
                                  regions=regions,
                                  nRunTrain=5,
                                  verbose=True,
                                  nRunFree=5,
                                  plotStatus=False)


# [curbd_arr, curbd_labels] = curbd.computeCURBD(model)